# 68. Speculative Decoding Benchmark | 投机解码基准
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `Speculative Decoding`, `基准对比` | **目标人群：** 项目决策练习者

---

## 本节导读

投机解码把一次生成拆成两个协作环节：草稿模型先提出一段候选，目标模型再验证这些候选。学习时要关注两个问题：候选能连续通过多少 token，以及减少的解码轮次能否抵消额外的 draft 与 verify 成本。本节沿着 baseline、接受与回退、端到端收益逐步完成判断。

**关键词：** `acceptance rate`, `draft cost`, `verify cost`, `benchmark`

---

## 前置阅读

**导语：** 进入本节前，先理解单 token 解码如何推进，以及 draft / target 如何协作。阅读时重点观察候选提出后哪些 token 被连续接受，以及一次验证是否真的减少了目标模型的重复解码工作。
- [21. Decoding Strategies | 解码策略](./21_Decoding_Strategies.ipynb)
- [23. Speculative Decoding | 投机解码](./23_Speculative_Decoding.ipynb)
- [35. Multi-Token Decoding | 多 Token 解码](./35_Multi_Token_Decoding.ipynb)
- [66. Inference Performance Comparison | 推理性能对比实验](./66_Inference_Performance_Comparison.ipynb)

### Step 1：理解 proposal、verify 与修正

投机解码让 draft 一次提出多个候选 token，target 再按顺序验证。target 从候选开头开始接受连续相同的 token；遇到第一个分歧时停止接受，写入 target 给出的 correction token，再进入下一轮 proposal。候选长度、draft 路径和验证成本可以不同，因此项目比较不能只看某一次的接受率。

| 轮内阶段 | 输入与状态变化 | 产生的机制证据 |
|:---|:---|:---|
| proposal | draft 提出一段候选 token | proposed tokens、proposal mode |
| verify | target 逐个比较候选与目标 token | 连续接受前缀长度 |
| accept | 开头连续相同的 token 一次推进 | accepted tokens、acceptance rate |
| correction | 首个分歧由 target token 修正 | rejected token、corrected tokens |
| 下一轮 | 新上下文继续生成候选 | round trace、每轮推进量 |

![投机解码 benchmark：从固定口径到项目决策](../docs/public/02_PyTorch_Algorithms/68_speculative_benchmark_flow.svg)

### Step 2：用 CPU 状态轨迹比较候选策略

CPU 练习把每轮 draft_tokens 与 target_tokens 作为已经给定的输入，逐轮计算最长连续接受前缀。可以为同一请求构造不同的 proposal 长度或 proposal mode，比较它们的接受、修正和成本轨迹；它不加载模型，但能先验证策略差异来自哪里。

| 输入或状态 | 规则 | 为什么重要 |
|:---|:---|:---|
| draft_tokens / target_tokens | 每轮都必须非空，按位置逐个比较 | 保证接受来自同一轮候选验证 |
| 连续接受 | 从第一个 token 开始；首个分歧后停止 | 不能跳过中间 miss 继续累计接受 |
| correction | 分歧时写入一个 target token | 保证本轮仍能推进上下文 |
| proposal mode | 同一 workload 下改变候选长度或生成路径 | 让接受与成本可以按策略比较 |
| round trace | 保存 proposal、接受前缀、拒绝 token 与 correction | 让聚合指标可以追溯回状态变化 |

### Step 3：从接受率走到策略净收益

高 acceptance rate 只说明 draft 候选较可靠；是否值得采用还取决于每轮推进量是否减少主要工作，以及 draft 与 verify 的额外成本是否抵消收益。先在相同 workload 下汇总不同 proposal mode，再与 target-only baseline 对照，才可以解释某项策略的取舍。

| 指标 | 计算或来源 | 判断时如何使用 |
|:---|:---|:---|
| acceptance rate | accepted tokens / proposed tokens | 候选的可靠程度 |
| accepted tokens per round | accepted tokens / rounds | 每轮真正推进多少 token |
| effective output tokens | accepted tokens + corrected tokens | CPU 机制模型中的总推进量 |
| draft cost | proposed tokens 乘以 draft 单 token 成本 | 候选越长并非总是越好 |
| verify cost | rounds 乘以每轮验证成本 | 用于检查验证是否抵消收益 |
| throughput speedup | candidate throughput / baseline throughput | 只有同 workload 记录才可解释为收益 |
| quality status | 输出一致性或任务指标 | 质量不通过时性能收益不进入决策 |

### Step 4：实现 CPU 投机机制与项目决策

题目区用三个机制 TODO 串起一项可比较的 CPU 项目：连续接受前缀、候选策略排序，以及质量与成本共同参与的决策。同 workload 汇总、baseline/candidate 对照和报告格式由骨架提供。

| TODO | 函数 | 机制责任 | 关键测试 |
|:---|:---|:---|:---|
| TODO 1 | longest_accepted_prefix | 从候选开头计算连续接受前缀 | 首个拒绝、全接受、不能跳过中间 miss |
| TODO 2 | speculative_rank_key | 按收益与验证代价排序质量合格的候选策略 | 排序方向、质量门槛、策略选择 |
| TODO 3 | choose_speculative_action | 用质量、接受率、吞吐与 verify 成本选择动作 | accept / tune / reject 与下一步 |
| 辅助函数 | simulate / summarize / compare / recommend | 状态记录、同口径汇总、差值计算和报告 | workload 契约、字段完整性 |


In [ ]:
from typing import Dict, List


In [ ]:
# 题目区保留三项会改变投机策略行为的机制 TODO：连续接受、候选策略排序和质量优先决策。
# 同 workload 汇总、baseline 对照和报告格式由骨架提供，避免把字段填写误当作机制实现。
def longest_accepted_prefix(draft_tokens, target_tokens):
    """返回从位置 0 开始、连续通过 target 验证的候选 token 数。

    返回首个分歧前的连续相同 token 数；后续偶然相同的 token 不计入。
    """
    if not draft_tokens or not target_tokens:
        raise ValueError('draft_tokens 和 target_tokens 必须非空')
    # TODO 1（连续接受前缀）：令 accepted 从 0 开始，逐对比较 draft_token / target_token。
    # 分歧时停止；相同时 accepted 增加 1，最后返回 accepted。
    # 变量提示：accepted、draft_token、target_token。
    raise NotImplementedError('TODO 1：请完成连续接受前缀机制')


def simulate_speculative_decode(rounds, draft_ms_per_token=1.0, verify_ms_per_round=4.0):
    """模拟一条 proposal/verify 路径的接受、修正与线性成本。

    rounds 的每项包含 draft_tokens 和 target_tokens；返回的轨迹和指标供后续策略比较使用。
    """
    if draft_ms_per_token < 0 or verify_ms_per_round < 0:
        raise ValueError('draft 和 verify 成本不能为负数')
    if not isinstance(rounds, list):
        raise TypeError('rounds 必须是 list')
    proposed_tokens = accepted_tokens = corrected_tokens = accepted_rounds = 0
    round_trace = []
    for round_index, item in enumerate(rounds):
        if not isinstance(item, dict) or 'draft_tokens' not in item or 'target_tokens' not in item:
            raise ValueError(f'第 {round_index} 轮必须包含 draft_tokens 和 target_tokens')
        draft, target = list(item['draft_tokens']), list(item['target_tokens'])
        accepted = longest_accepted_prefix(draft, target)
        proposed_tokens += len(draft)
        accepted_tokens += accepted
        accepted_rounds += int(accepted == len(draft))
        corrected = int(accepted < len(draft))
        corrected_tokens += corrected
        rejected_token = target[accepted] if accepted < len(target) and accepted < len(draft) else None
        round_trace.append({'round': round_index, 'proposed': len(draft), 'accepted_prefix_length': accepted,
                            'accepted': accepted, 'rejected_token': rejected_token, 'corrected': corrected})
    round_count = len(rounds)
    accepted_tokens_per_round = accepted_tokens / round_count if round_count else 0.0
    draft_cost_ms = proposed_tokens * draft_ms_per_token
    verify_cost_ms = round(round_count * verify_ms_per_round, 4)
    return {'rounds': round_count, 'proposed_tokens': proposed_tokens, 'accepted_tokens': accepted_tokens,
            'corrected_tokens': corrected_tokens, 'effective_output_tokens': accepted_tokens + corrected_tokens,
            'acceptance_rate': accepted_tokens / proposed_tokens if proposed_tokens else 0.0,
            'accepted_tokens_per_round': accepted_tokens_per_round, 'draft_cost_ms': draft_cost_ms,
            'verify_cost_ms': verify_cost_ms, 'fully_accepted_rounds': accepted_rounds, 'round_trace': round_trace,
            'mechanism_metrics': {'proposed_tokens': proposed_tokens, 'accepted_tokens': accepted_tokens,
                                  'corrected_tokens': corrected_tokens,
                                  'acceptance_rate': accepted_tokens / proposed_tokens if proposed_tokens else 0.0,
                                  'accepted_tokens_per_round': accepted_tokens_per_round},
            'cost_model_metrics': {'draft_cost_ms': draft_cost_ms, 'verify_cost_ms': verify_cost_ms}}


def summarize_speculative_benchmark(runs: List[Dict[str, float]]) -> Dict[str, object]:
    """按 proposal mode 汇总同一 workload 的策略证据。"""
    if not runs:
        return {'run_count': 0, 'workload_id': None, 'strategy_summary': {}, 'best_throughput_run': None}
    required = {'name', 'workload_id', 'proposal_mode', 'acceptance_rate', 'throughput', 'verify_cost_ms'}
    missing = [sorted(required - set(item)) for item in runs if not required.issubset(item)]
    if missing:
        raise KeyError(f'每条 run 必须包含 {sorted(required)}，缺失字段：{missing}')
    workload_ids = {item['workload_id'] for item in runs}
    if len(workload_ids) != 1:
        raise ValueError('策略汇总必须使用同一 workload_id')
    grouped = {}
    for item in runs:
        grouped.setdefault(item['proposal_mode'], []).append(item)
    strategy_summary = {
        mode: {'run_count': len(items),
               'avg_acceptance_rate': round(sum(item['acceptance_rate'] for item in items) / len(items), 6),
               'avg_throughput': round(sum(item['throughput'] for item in items) / len(items), 6),
               'avg_verify_cost_ms': round(sum(item['verify_cost_ms'] for item in items) / len(items), 6)}
        for mode, items in grouped.items()
    }
    best = max(runs, key=lambda item: item['throughput'])
    return {'run_count': len(runs), 'workload_id': workload_ids.pop(), 'strategy_summary': strategy_summary,
            'best_throughput_run': best['name']}


def compare_speculative_to_baseline(baseline: Dict[str, float], candidate: Dict[str, float]) -> Dict[str, float]:
    """计算 candidate 相对同一 workload 的 G0 baseline 的指标变化。"""
    required = {'workload_id', 'ttft_ms', 'throughput', 'acceptance_rate', 'verify_cost_ms'}
    for label, record in [('baseline', baseline), ('candidate', candidate)]:
        missing = sorted(required - set(record))
        if missing:
            raise KeyError(f'{label} 缺少比较字段：{missing}')
    if baseline['workload_id'] != candidate['workload_id']:
        raise ValueError('baseline 与 candidate 必须使用同一 workload_id')
    if baseline['throughput'] <= 0:
        raise ValueError('baseline throughput 必须大于 0')
    return {'ttft_delta_ms': candidate['ttft_ms'] - baseline['ttft_ms'],
            'throughput_gain': candidate['throughput'] - baseline['throughput'],
            'acceptance_rate': candidate['acceptance_rate'],
            'verify_cost_delta': candidate['verify_cost_ms'] - baseline['verify_cost_ms'],
            'throughput_speedup': candidate['throughput'] / baseline['throughput']}


def speculative_rank_key(candidate):
    """返回质量合格候选的升序排序 key。"""
    summary = candidate['comparison']
    # TODO 2（候选策略排序）：负号让更高吞吐收益和接受率排在前面；较低 verify 成本优先。
    # rank_key = ???  # 顺序：throughput_gain、acceptance_rate、verify_cost_delta、ttft_delta_ms。
    raise NotImplementedError('TODO 2：请定义候选策略排序机制')


def select_speculative_candidate(candidates):
    """先排除质量不合格候选，再按投机收益机制排序。

    每个候选需要 name、quality_ok 和 comparison；返回优先进入复测的候选。
    """
    eligible = [candidate for candidate in candidates if candidate['quality_ok']]
    eligible.sort(key=speculative_rank_key)
    return {'candidate_count': len(candidates), 'eligible_count': len(eligible),
            'selected': eligible[0] if eligible else None,
            'rejected_names': [candidate['name'] for candidate in candidates if not candidate['quality_ok']]}


def choose_speculative_action(comparison, quality_ok, min_acceptance_rate, max_verify_cost_delta):
    """将质量、接受率、吞吐与验证成本收成策略动作。

    返回值只能是 accept、tune 或 reject；质量失败直接拒绝。
    """
    acceptance_ok = comparison['acceptance_rate'] >= min_acceptance_rate
    throughput_ok = comparison['throughput_gain'] > 0
    verify_cost_ok = comparison['verify_cost_delta'] <= max_verify_cost_delta
    # TODO 3（策略决策）：质量先作为硬门槛；三项收益全通过为 accept，部分通过为 tune，其余 reject。
    # 变量提示：quality_ok、acceptance_ok、throughput_ok、verify_cost_ok。
    raise NotImplementedError('TODO 3：请完成投机策略决策机制')


def recommend_speculative_run(baseline, candidate, min_acceptance_rate, quality_ok=True, max_verify_cost_delta=10.0):
    """将已选择的策略动作转换为项目报告；报告字段不作为题目挖空。"""
    if not 0 <= min_acceptance_rate <= 1 or max_verify_cost_delta < 0:
        raise ValueError('决策阈值必须处于有效范围')
    comparison = compare_speculative_to_baseline(baseline, candidate)
    decision = choose_speculative_action(comparison, quality_ok, min_acceptance_rate, max_verify_cost_delta)
    details = {
        'accept': ('吞吐收益、接受率和验证成本都达标', 'promote_to_serving_eval'),
        'tune': ('存在收益，但接受率或验证成本仍需调优', 'refine_draft_or_verify'),
        'reject': ('质量、接受率或吞吐门槛未通过', 'fallback_to_baseline'),
    }
    reason, next_action = details[decision]
    return {'decision': decision, 'reason': reason, 'next_action': next_action}


In [ ]:
# 本测试区验证 CPU 的连续接受、候选策略排序与质量门槛决策。
# GPU/backend 的真实性能与输出质量在 Step 5 单独验证，不作为 CPU 题目区的通过条件。
def _build_speculative_fixture():
    """构造一轮分歧和一轮全接受，覆盖两种关键状态。"""
    return [{'draft_tokens': [1, 2, 3], 'target_tokens': [1, 2, 9]},
            {'draft_tokens': [4, 5], 'target_tokens': [4, 5]}]


def _record(name, mode, ttft_ms, throughput, acceptance, verify, quality=True):
    """构造同 workload 的候选记录；所有比较指标显式写出。"""
    return {'name': name, 'workload_id': 'shared_prompts', 'proposal_mode': mode,
            'ttft_ms': ttft_ms, 'throughput': throughput,
            'acceptance_rate': acceptance, 'verify_cost_ms': verify, 'quality_ok': quality}


def test_continuous_acceptance_prefix():
    """验证首个分歧、全接受、首 token 拒绝和较短 target。"""
    assert longest_accepted_prefix([1, 2, 3], [1, 2, 9]) == 2
    assert longest_accepted_prefix([1, 2], [1, 2]) == 2
    assert longest_accepted_prefix([1, 2], [9, 2]) == 0
    assert longest_accepted_prefix([1, 2, 3], [1, 2]) == 2


def test_state_transition():
    """验证接受前缀正确形成 correction 与逐轮 trace。"""
    simulation = simulate_speculative_decode(_build_speculative_fixture())
    assert (simulation['accepted_tokens'], simulation['corrected_tokens']) == (4, 1)
    assert simulation['round_trace'][0]['accepted_prefix_length'] == 2
    assert simulation['round_trace'][0]['rejected_token'] == 9


def test_strategy_summary_and_ranking():
    """验证相同 workload 下按收益和代价排序候选 proposal mode。"""
    short = _record('short', 'short', 109, 110, .60, 42)
    long = _record('long', 'long', 106, 135, .72, 48)
    summary = summarize_speculative_benchmark([short, long])
    assert set(summary['strategy_summary']) == {'short', 'long'}
    baseline = _record('g0', 'target_only', 120, 100, 0., 40)
    candidates = [{'name': item['name'], 'quality_ok': item['quality_ok'],
                   'comparison': compare_speculative_to_baseline(baseline, item)} for item in [short, long]]
    assert select_speculative_candidate(candidates)['selected']['name'] == 'long'


def test_workload_and_quality_guards():
    """验证 workload 契约和质量门槛会阻止无效候选被选中。"""
    short = _record('short', 'short', 109, 110, .60, 42)
    try:
        summarize_speculative_benchmark([short, {**short, 'workload_id': 'other'}])
    except ValueError:
        pass
    else:
        raise AssertionError('混合 workload 不应进入策略汇总')
    baseline = _record('g0', 'target_only', 120, 100, 0., 40)
    comparison = compare_speculative_to_baseline(baseline, short)
    rejected = {'name': 'invalid-quality', 'quality_ok': False, 'comparison': {**comparison, 'throughput_gain': 999}}
    assert select_speculative_candidate([rejected])['selected'] is None
    assert choose_speculative_action(comparison, False, .5, 10) == 'reject'
    assert choose_speculative_action(comparison, True, .5, 10) == 'accept'
    assert choose_speculative_action({**comparison, 'verify_cost_delta': 20}, True, .5, 10) == 'tune'


def test_speculative_benchmark_template():
    """运行本节全部 CPU 机制测试。"""
    test_continuous_acceptance_prefix()
    test_state_transition()
    test_strategy_summary_and_ranking()
    test_workload_and_quality_guards()
    print('测试通过：投机解码的连续接受、候选排序与策略决策机制均通过。')


try:
    test_speculative_benchmark_template()
except NotImplementedError:
    print('请先完成 TODO 部分的代码！')
    raise


## 参考代码与解析


In [ ]:
# 题目区保留三项会改变投机策略行为的机制 TODO：连续接受、候选策略排序和质量优先决策。
# 同 workload 汇总、baseline 对照和报告格式由骨架提供，避免把字段填写误当作机制实现。
def longest_accepted_prefix(draft_tokens, target_tokens):
    """返回从位置 0 开始、连续通过 target 验证的候选 token 数。"""
    if not draft_tokens or not target_tokens:
        raise ValueError('draft_tokens 和 target_tokens 必须非空')
    # TODO 1：首个分歧后立即结束，因此不会把后续偶然相同的 token 计入接受前缀。
    accepted = 0
    for draft_token, target_token in zip(draft_tokens, target_tokens):
        if draft_token != target_token:
            break
        accepted += 1
    return accepted


def simulate_speculative_decode(rounds, draft_ms_per_token=1.0, verify_ms_per_round=4.0):
    """模拟一条 proposal/verify 路径的接受、修正与线性成本。"""
    if draft_ms_per_token < 0 or verify_ms_per_round < 0:
        raise ValueError('draft 和 verify 成本不能为负数')
    if not isinstance(rounds, list):
        raise TypeError('rounds 必须是 list')
    proposed_tokens = accepted_tokens = corrected_tokens = accepted_rounds = 0
    round_trace = []
    for round_index, item in enumerate(rounds):
        if not isinstance(item, dict) or 'draft_tokens' not in item or 'target_tokens' not in item:
            raise ValueError(f'第 {round_index} 轮必须包含 draft_tokens 和 target_tokens')
        draft, target = list(item['draft_tokens']), list(item['target_tokens'])
        accepted = longest_accepted_prefix(draft, target)
        proposed_tokens += len(draft)
        accepted_tokens += accepted
        accepted_rounds += int(accepted == len(draft))
        corrected = int(accepted < len(draft))
        corrected_tokens += corrected
        rejected_token = target[accepted] if accepted < len(target) and accepted < len(draft) else None
        round_trace.append({'round': round_index, 'proposed': len(draft), 'accepted_prefix_length': accepted,
                            'accepted': accepted, 'rejected_token': rejected_token, 'corrected': corrected})
    round_count = len(rounds)
    accepted_tokens_per_round = accepted_tokens / round_count if round_count else 0.0
    draft_cost_ms = proposed_tokens * draft_ms_per_token
    verify_cost_ms = round(round_count * verify_ms_per_round, 4)
    return {'rounds': round_count, 'proposed_tokens': proposed_tokens, 'accepted_tokens': accepted_tokens,
            'corrected_tokens': corrected_tokens, 'effective_output_tokens': accepted_tokens + corrected_tokens,
            'acceptance_rate': accepted_tokens / proposed_tokens if proposed_tokens else 0.0,
            'accepted_tokens_per_round': accepted_tokens_per_round, 'draft_cost_ms': draft_cost_ms,
            'verify_cost_ms': verify_cost_ms, 'fully_accepted_rounds': accepted_rounds, 'round_trace': round_trace,
            'mechanism_metrics': {'proposed_tokens': proposed_tokens, 'accepted_tokens': accepted_tokens,
                                  'corrected_tokens': corrected_tokens,
                                  'acceptance_rate': accepted_tokens / proposed_tokens if proposed_tokens else 0.0,
                                  'accepted_tokens_per_round': accepted_tokens_per_round},
            'cost_model_metrics': {'draft_cost_ms': draft_cost_ms, 'verify_cost_ms': verify_cost_ms}}


def summarize_speculative_benchmark(runs: List[Dict[str, float]]) -> Dict[str, object]:
    """按 proposal mode 汇总同一 workload 的策略证据。"""
    if not runs:
        return {'run_count': 0, 'workload_id': None, 'strategy_summary': {}, 'best_throughput_run': None}
    required = {'name', 'workload_id', 'proposal_mode', 'acceptance_rate', 'throughput', 'verify_cost_ms'}
    missing = [sorted(required - set(item)) for item in runs if not required.issubset(item)]
    if missing:
        raise KeyError(f'每条 run 必须包含 {sorted(required)}，缺失字段：{missing}')
    workload_ids = {item['workload_id'] for item in runs}
    if len(workload_ids) != 1:
        raise ValueError('策略汇总必须使用同一 workload_id')
    grouped = {}
    for item in runs:
        grouped.setdefault(item['proposal_mode'], []).append(item)
    strategy_summary = {
        mode: {'run_count': len(items),
               'avg_acceptance_rate': round(sum(item['acceptance_rate'] for item in items) / len(items), 6),
               'avg_throughput': round(sum(item['throughput'] for item in items) / len(items), 6),
               'avg_verify_cost_ms': round(sum(item['verify_cost_ms'] for item in items) / len(items), 6)}
        for mode, items in grouped.items()
    }
    best = max(runs, key=lambda item: item['throughput'])
    return {'run_count': len(runs), 'workload_id': workload_ids.pop(), 'strategy_summary': strategy_summary,
            'best_throughput_run': best['name']}


def compare_speculative_to_baseline(baseline: Dict[str, float], candidate: Dict[str, float]) -> Dict[str, float]:
    """计算 candidate 相对同一 workload 的 G0 baseline 的指标变化。"""
    required = {'workload_id', 'ttft_ms', 'throughput', 'acceptance_rate', 'verify_cost_ms'}
    for label, record in [('baseline', baseline), ('candidate', candidate)]:
        missing = sorted(required - set(record))
        if missing:
            raise KeyError(f'{label} 缺少比较字段：{missing}')
    if baseline['workload_id'] != candidate['workload_id']:
        raise ValueError('baseline 与 candidate 必须使用同一 workload_id')
    if baseline['throughput'] <= 0:
        raise ValueError('baseline throughput 必须大于 0')
    return {'ttft_delta_ms': candidate['ttft_ms'] - baseline['ttft_ms'],
            'throughput_gain': candidate['throughput'] - baseline['throughput'],
            'acceptance_rate': candidate['acceptance_rate'],
            'verify_cost_delta': candidate['verify_cost_ms'] - baseline['verify_cost_ms'],
            'throughput_speedup': candidate['throughput'] / baseline['throughput']}


def speculative_rank_key(candidate):
    """返回质量合格候选的升序排序 key。"""
    summary = candidate['comparison']
    # TODO 2：更大的吞吐收益、接受率和更小的 verify / TTFT 代价优先。
    return (-summary['throughput_gain'], -summary['acceptance_rate'],
            summary['verify_cost_delta'], summary['ttft_delta_ms'])


def select_speculative_candidate(candidates):
    """先排除质量不合格候选，再按投机收益机制排序。"""
    eligible = [candidate for candidate in candidates if candidate['quality_ok']]
    eligible.sort(key=speculative_rank_key)
    return {'candidate_count': len(candidates), 'eligible_count': len(eligible),
            'selected': eligible[0] if eligible else None,
            'rejected_names': [candidate['name'] for candidate in candidates if not candidate['quality_ok']]}


def choose_speculative_action(comparison, quality_ok, min_acceptance_rate, max_verify_cost_delta):
    """将质量、接受率、吞吐与验证成本收成策略动作。"""
    acceptance_ok = comparison['acceptance_rate'] >= min_acceptance_rate
    throughput_ok = comparison['throughput_gain'] > 0
    verify_cost_ok = comparison['verify_cost_delta'] <= max_verify_cost_delta
    # TODO 3：质量失败直接回退；三项收益通过才接受，部分收益进入调优。
    if not quality_ok:
        return 'reject'
    if acceptance_ok and throughput_ok and verify_cost_ok:
        return 'accept'
    if throughput_ok and (acceptance_ok or verify_cost_ok):
        return 'tune'
    return 'reject'


def recommend_speculative_run(baseline, candidate, min_acceptance_rate, quality_ok=True, max_verify_cost_delta=10.0):
    """将已选择的策略动作转换为项目报告；报告字段不作为题目挖空。"""
    if not 0 <= min_acceptance_rate <= 1 or max_verify_cost_delta < 0:
        raise ValueError('决策阈值必须处于有效范围')
    comparison = compare_speculative_to_baseline(baseline, candidate)
    decision = choose_speculative_action(comparison, quality_ok, min_acceptance_rate, max_verify_cost_delta)
    details = {
        'accept': ('吞吐收益、接受率和验证成本都达标', 'promote_to_serving_eval'),
        'tune': ('存在收益，但接受率或验证成本仍需调优', 'refine_draft_or_verify'),
        'reject': ('质量、接受率或吞吐门槛未通过', 'fallback_to_baseline'),
    }
    reason, next_action = details[decision]
    return {'decision': decision, 'reason': reason, 'next_action': next_action}


### 解析

投机解码的比较从一轮 token 状态开始：先得到连续接受前缀，再把不同 proposal mode 放到同一请求分布下比较，最后依据质量和成本选择下一步。

**TODO 1：连续接受前缀**

- 从候选开头逐个比较；首个分歧后停止，后续偶然相同的 token 不会被误计。
- 前缀长度会直接影响 correction、round trace 和 acceptance rate。

**TODO 2：候选策略排序**

- 质量不合格候选先被排除；其余候选依次比较吞吐收益、接受率、verify 成本和 TTFT。
- 排序得到的是优先复测的 proposal mode，不是最终部署结论。

**TODO 3：质量优先的策略动作**

- 质量失败直接 `reject`；吞吐、接受率和 verify 成本都达标才 `accept`。
- 有收益但仍需调整 proposal 或验证代价时返回 `tune`，再在 Step 5 记录真实 backend 结果。


### Step 5（可选）：GPU 与 backend 实验——真实 speculative 对照


#### 5.1 环境、输入与固定条件

先固定 target / draft、生成条件和测量协议，再记录本次实际使用的 GPU、dtype 与 backend。G0、G1、G2 使用同一请求集和生成条件；G2 每次只改变一个候选生成因素。

| 类别 | 固定项 | 本轮默认值 | 作用 |
|:---|:---|:---|:---|
| 模型 | target / draft | Qwen2.5 1.5B / 0.5B | 形成可复现的 G0/G1 对照 |
| 生成 | max tokens / temperature / top-p | 64 / 0 / 1 | 固定输出长度和采样行为 |
| 策略 | proposal length | 5；G2 改为 3 或 8 | 单独观察候选预算的影响 |
| 测量 | prompts / warmup / repeats | 30 / 5 / 3 | 减少单次请求带来的偶然波动 |
| 环境 | GPU / dtype / backend | 运行时写入结果 JSON | 不同硬件分别保留证据 |

![GPU speculative benchmark：从环境到项目决策](../docs/public/02_PyTorch_Algorithms/68_speculative_gpu_flow.svg)


#### 5.2 环境启动检查

确认 Notebook 识别到 CUDA，并记录当前设备与 PyTorch 版本；这些信息会与后续 JSON 一起帮助定位模型、驱动或 backend 启动问题。


In [ ]:
import sys
import torch

print({'python': sys.executable, 'torch': torch.__version__, 'cuda_available': torch.cuda.is_available()})
if torch.cuda.is_available():
    print({'device': torch.cuda.get_device_name(0), 'capability': torch.cuda.get_device_capability(0)})
else:
    print('当前环境没有 CUDA；可以继续阅读 CPU 机制结果，但不能执行 GPU/backend 实验。')


#### 5.3 配置实验条件

下面的字段构成每条 G0/G1/G2 记录的共同契约。`G0` 只运行 target，`G1` 启用 draft + target，`G2` 在保持其余字段一致的前提下只改变 proposal length、draft family 或候选预算。

| 字段组 | 必须记录的内容 | 本节用途 |
|:---|:---|:---|
| 身份与角色 | `project`、`role`、`strategy` | 区分 baseline 与 candidate |
| workload / config | 模型、backend、dtype、输入输出长度、并发、seed、repeat | 保证 G0/G1/G2 可比较 |
| 端到端指标 | TTFT、TPOT、throughput、P95/P99、peak memory、queue wait | 判断服务收益与尾延迟 |
| 投机机制 | acceptance、每轮推进、target calls、draft / verify cost | 解释收益来自何处 |
| 候选生成 | `draft_family`、`proposal_mode`、`confidence_threshold`、budget | 标识唯一改变的策略因素 |
| 质量与状态 | 输出一致性、任务质量、`ok` / `unsupported` / `OOM` / `failed` | 保留可用性和失败证据 |
| 证据与决策 | `evidence_level`、`decision`、failure reason、retest path | 支持复测和项目收口 |


In [ ]:
try:
    from tools.inference_project_runtime import locate_repo_root
    REPO_ROOT = locate_repo_root()
    from tools.inference_project_runtime import (
        shared_project_config, save_project_result, start_optional_vllm,
        start_speculative_vllm, stop_optional_vllm, run_backend_benchmark,
    )
except ModuleNotFoundError:
    def shared_project_config(**kwargs): return kwargs
    def save_project_result(*args, **kwargs): raise RuntimeError('需要从仓库根目录运行真实 backend 入口')

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # G0/G1 的 target；第一轮正式候选。
DRAFT_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # 与 target 同系列的 draft。
PROPOSAL_LENGTH = 5  # 每轮 draft 提议的 token 数；G2 再改为 3 / 8。
DRAFT_FAMILY = 'autoregressive_draft'  # 可选：autoregressive_draft / multi_token / dflash / dspark。
PROPOSAL_MODE = 'tokenwise'  # 可选：tokenwise / block / semi_autoregressive。
CONFIDENCE_THRESHOLD = None  # DSpark 类路径才填写；未启用时保持 None。
MIN_ACCEPTANCE_RATE = 0.6  # 教学决策门槛，不是通用生产阈值。
NUM_PROMPTS = 30  # 正式 benchmark 下限；链路 smoke 可临时使用更小值。
WARMUP = 5  # 正式测量前的预热请求数。
REPEATS = 3  # 同一配置的重复次数。
MAX_TOKENS = 64  # G0/G1/G2 必须保持一致。
TEMPERATURE = 0.0  # 确定性生成，便于比较输出质量。
TOP_P = 1.0  # 与 temperature 一起固定采样口径。
HARDWARE_PROFILE = 'auto'  # auto / H0_12GB_5070Ti_Laptop / H1_24GB_RTX4090。
EXPECTED_GPU_MEMORY_GB = {'H0_12GB_5070Ti_Laptop': 12, 'H1_24GB_RTX4090': 24}
RUN_ID = 'smoke'  # 正式复测时替换为可追溯的日期或实验批次标识。
RESULT_PATH = f'benchmarks/results/68_g1_speculative_vllm_{RUN_ID}.json'  # candidate 结果文件。
MANIFEST_PATH = f'benchmarks/results/68_manifest_{RUN_ID}.json'  # G0/G1/G2 对照清单。
project_config = shared_project_config(
    model=MODEL_ID, backend='vllm', dtype='auto', generated_tokens=MAX_TOKENS,
    cache_policy='default', draft_model=DRAFT_MODEL_ID,
    proposal_length=PROPOSAL_LENGTH, min_acceptance_rate=MIN_ACCEPTANCE_RATE,
    draft_family=DRAFT_FAMILY, proposal_mode=PROPOSAL_MODE, confidence_threshold=CONFIDENCE_THRESHOLD,
    num_prompts=NUM_PROMPTS, warmup=WARMUP, repeats=REPEATS,
    temperature=TEMPERATURE, top_p=TOP_P, hardware_profile=HARDWARE_PROFILE,
)
print(project_config)
RUN_BACKEND_SMOKE = False  # 仅验证 baseline endpoint，不等于 speculative 已启用。
RUN_REAL_SPECULATIVE = False  # 当前 helper 不会自动启用 speculative backend。
def validate_speculative_config():
    """在启动 backend 前检查 G0/G1 的必要配置。"""
    if not MODEL_ID or not DRAFT_MODEL_ID:
        return ['target 和 draft model 都必须填写']
    if MODEL_ID == DRAFT_MODEL_ID:
        return ['target 和 draft model 不能相同']
    if PROPOSAL_LENGTH <= 0 or NUM_PROMPTS <= 0 or WARMUP < 0 or REPEATS <= 0:
        return ['proposal_length、num_prompts、repeats 必须为正数，warmup 不能为负数']
    if not 0 <= TEMPERATURE:
        return ['temperature 不能为负数']
    return []

config_errors = validate_speculative_config()
if config_errors:
    raise ValueError('speculative 配置无效：' + '；'.join(config_errors))
def build_experiment_plan():
    """生成正式采集前可复核的 G0/G1/G2 计划，不启动 backend。"""
    return [
        {'group': 'G0', 'strategy': 'target_baseline', 'enabled': False,
         'proposal_length': None, 'draft_model': None},
        {'group': 'G1', 'strategy': 'speculative', 'enabled': False,
         'proposal_length': PROPOSAL_LENGTH, 'draft_model': DRAFT_MODEL_ID},
        *[{'group': f'G2_len_{length}', 'strategy': 'speculative', 'enabled': False,
           'proposal_length': length, 'draft_model': DRAFT_MODEL_ID}
          for length in (3, 8)],
    ]

def summarize_output_quality(outputs, references=None):
    """汇总请求成功率和可选的 exact-match；不替代任务评测。"""
    outputs = list(outputs or [])
    success = [item for item in outputs if isinstance(item, str) and item.strip()]
    result = {'total': len(outputs), 'non_empty': len(success),
              'success_rate': len(success) / len(outputs) if outputs else 0.0}
    if references is not None:
        references = list(references)
        if len(references) != len(outputs):
            raise ValueError('references 与 outputs 长度必须一致')
        result['exact_match_rate'] = (
            sum(output == reference for output, reference in zip(outputs, references)) / len(outputs)
            if outputs else 0.0
        )
    return result

experiment_plan = build_experiment_plan()
print({'experiment_plan': experiment_plan})


#### 5.4 执行实验并保存 JSON

先采集 G0 target-only baseline；再探测并启动 G1 draft + target 路径；最后以单个变量构造 G2。每次执行都保存独立 JSON，不能运行的路径也要保留状态和原因。

| 执行分支 | 当前代码会做什么 | 产物 |
|:---|:---|:---|
| G0 smoke | 启动 target-only vLLM 并采集基线 | `68_g0_target_baseline_...json` |
| G1 capability | 探测 speculative 参数并尝试启动 draft + target | `68_g1_speculative_...json` 或 unsupported 状态 |
| G2 单变量对照 | 在真实 adapter 可用后分别修改 proposal length 或 draft 配置 | 独立的 G2 JSON |
| manifest | 汇总本次已执行结果的位置和状态 | `68_manifest_...json` |


In [ ]:
if RUN_REAL_SPECULATIVE and not DRAFT_MODEL_ID:
    raise ValueError('真实 speculative 实验必须先配置 DRAFT_MODEL_ID。')
if RUN_REAL_SPECULATIVE:
    # 先探测 CLI 能力；不支持时保存 unsupported 报告，不加载模型。
    from tools.backend_runtime import probe_vllm_speculative_support
    capability = probe_vllm_speculative_support()
    if capability['status'] != 'supported':
        save_project_result(
            RESULT_PATH, project='68', strategy='speculative', config=project_config,
            metrics={}, quality={'status': 'unsupported', 'speculative_enabled': False},
            decision={'decision': 'tune', 'reason': '当前 vLLM CLI 未发现可识别的 speculative 参数'},
            strategy_metrics={'evidence_level': 'backend_capability_probe',
                              'capability': capability},
        )
        print({'speculative_capability': capability, 'status': 'unsupported'})
    else:
        server, log_path, port, selected_dtype, target_path, capability = start_speculative_vllm(
            target_model_id=MODEL_ID, draft_model_id=DRAFT_MODEL_ID,
            dtype='auto', proposal_length=PROPOSAL_LENGTH,
            served_model_name=MODEL_ID,
        )
        try:
            print({'speculative_capability': capability, 'port': port, 'dtype': selected_dtype})
            save_project_result(
                RESULT_PATH, project='68', strategy='speculative', config=project_config,
                metrics={}, quality={'status': 'not_evaluated', 'speculative_enabled': True},
                decision={'decision': 'tune', 'reason': 'backend 已启动，但尚未运行 G1/G2 benchmark'},
                strategy_metrics={'evidence_level': 'backend_started_no_metrics',
                                  'capability': capability},
            )
        finally:
            stop_optional_vllm(server, log_path)
if RUN_BACKEND_SMOKE:
    server, log_path, port, selected_dtype, model_path = start_optional_vllm(
        model_id=MODEL_ID, model_source='auto', dtype='auto',
        served_model_name=MODEL_ID,
    )
    try:
        report = run_backend_benchmark(
            project='68', base_url=f'http://127.0.0.1:{port}', model=MODEL_ID,
            label='vllm-baseline-for-speculative',
            output='benchmarks/results/68_backend_smoke.json',
            dtype=selected_dtype,
        )
        normalized = report.get('normalized_result', {})
        save_project_result(
            f'benchmarks/results/68_g0_target_baseline_vllm_{RUN_ID}.json',
            project='68', strategy='target_baseline', config=project_config,
            metrics=normalized.get('metrics', report.get('metrics', {})),
            quality={'status': 'reference_only', 'speculative_enabled': False},
            decision={'decision': 'measure', 'reason': 'G0 target baseline only'},
            strategy_metrics={'speculative_enabled': False, 'evidence_level': 'gpu_baseline_smoke'},
        )
    finally:
        stop_optional_vllm(server, log_path)
if RUN_REAL_SPECULATIVE or RUN_BACKEND_SMOKE:
    manifest = {
        'schema_version': 'inference-project-manifest/v1',
        'project': '68',
        'experiment': 'speculative_decoding',
        'config': project_config,
        'artifact': {'manifest_path': MANIFEST_PATH, 'candidate_result_path': RESULT_PATH, 'baseline_result_path': f'benchmarks/results/68_g0_target_baseline_vllm_{RUN_ID}.json'},
        'evidence_level': 'real_backend_smoke' if RUN_REAL_SPECULATIVE else 'gpu_baseline_smoke',
        'failure': None,
        'decision': {'decision': 'tune', 'reason': 'manifest 仅汇总已执行的 G0/G1 结果，需读取 strategy_metrics 后再判断'},
    }
    manifest_path = Path(MANIFEST_PATH)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'实验清单已保存: {manifest_path}')

# G1 接入真实 draft/target adapter 后，必须提供真实 metrics 再保存；strategy_metrics
# 至少记录 acceptance_rate、accepted_tokens_per_round、target_forward_calls、
# draft_cost_ms 和 verify_cost_ms，不能只保存 TTFT 或吞吐：
# save_project_result(RESULT_PATH, project='68', strategy='speculative',
#     config=project_config, metrics=metrics, quality=quality,
#     strategy_metrics={'acceptance_rate': acceptance_rate,
#                       'accepted_tokens_per_round': accepted_tokens_per_round,
#                       'target_forward_calls': target_forward_calls,
#                       'draft_cost_ms': draft_cost_ms,
#                       'verify_cost_ms': verify_cost_ms,
#                       'draft_family': DRAFT_FAMILY, 'proposal_mode': PROPOSAL_MODE,
#                       'confidence_threshold': CONFIDENCE_THRESHOLD},
#     decision=decision)

#### 5.5 读取结果与记录证据

读取 manifest 后，逐条打开已保存的 G0/G1/G2 JSON，核对共同配置、机制字段、端到端指标和运行状态。下表是结果汇总时保留的最小阅读视图；详细字段仍以 JSON 为准。

| 组别 | 对照变化与配置 | 机制证据 | 端到端与质量 | 证据与下一步 |
|:---|:---|:---|:---|:---|
| G0 | target-only；固定 workload | 不适用 | TTFT / TPOT / throughput / memory；参考输出 | baseline evidence；作为对照 |
| G1 | draft + target；固定 proposal | acceptance、每轮推进、target calls、verify cost | 与 G0 同口径；质量状态 | 若字段齐全，进入策略判断 |
| G2 | 只改变一项候选因素 | 与 G1 相同 | 与 G0 同口径；质量状态 | 判断该变量是否值得保留 |
| 失败记录 | unsupported / OOM / failed | 可用时保留已采集机制字段 | failure reason、retest path | 不与成功运行混合平均 |


In [ ]:
from pathlib import Path
import json

RESULT_CONFIG_FIELDS = ('model', 'backend', 'dtype', 'generated_tokens', 'concurrency')
RESULT_METRIC_FIELDS = ('ttft_ms', 'tpot_ms', 'throughput', 'p95_ms', 'p99_ms', 'peak_memory_mb', 'queue_wait_ms')
SPECULATIVE_FIELDS = ('acceptance_rate', 'accepted_tokens_per_round', 'target_forward_calls', 'draft_cost_ms', 'verify_cost_ms')

def read_result_record(result_path):
    """读取一条项目 JSON，提取配置、端到端指标、投机机制字段和状态。"""
    result_path = Path(result_path)
    if not result_path.exists():
        return {'path': str(result_path), 'status': 'missing'}
    record = json.loads(result_path.read_text(encoding='utf-8'))
    config = record.get('config', {})
    metrics = record.get('metrics', {})
    strategy_metrics = record.get('strategy_metrics', {})
    return {
        'path': str(result_path),
        'status': record.get('status', 'not_recorded'),
        'role': record.get('role'),
        'strategy': record.get('strategy'),
        'config': {name: config.get(name) for name in RESULT_CONFIG_FIELDS},
        'metrics': {name: metrics.get(name) for name in RESULT_METRIC_FIELDS},
        'strategy_metrics': {name: strategy_metrics.get(name) for name in SPECULATIVE_FIELDS},
        'quality': record.get('quality', {}),
        'evidence_level': record.get('evidence_level'),
        'decision': record.get('decision', {}),
        'failure': record.get('failure', {}),
    }


def read_project_evidence(manifest_path):
    """读取 manifest 引用的结果，并检查共同配置是否足以进行组内比较。"""
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        return {'status': 'not_run', 'manifest': str(manifest_path), 'results': []}
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    artifact = manifest.get('artifact', {})
    result_paths = [artifact.get('baseline_result_path'), artifact.get('candidate_result_path')]
    result_paths.extend(artifact.get('g2_result_paths', []))
    results = [read_result_record(item) for item in result_paths if item]
    readable = [item for item in results if item.get('status') != 'missing']
    config_signatures = {tuple(item['config'].get(name) for name in RESULT_CONFIG_FIELDS) for item in readable}
    return {
        'status': 'ok', 'manifest': str(manifest_path), 'results': results,
        'shared_config_ok': len(config_signatures) <= 1,
        'evidence_level': manifest.get('evidence_level'),
        'decision': manifest.get('decision'),
    }


evidence_report = read_project_evidence(globals().get('MANIFEST_PATH', 'benchmarks/results/68_manifest_smoke.json'))
print(json.dumps(evidence_report, ensure_ascii=False, indent=2))


#### 5.6 解释结果与形成决策

先确认 G1 是否在质量通过的前提下优于 G0，再用 G2 观察 proposal length 或 draft 配置带来的变化。将机制证据、端到端指标和失败状态一起阅读，才能决定下一次实验。

| 观察到的结果 | 决策 | 下一步 |
|:---|:---|:---|
| G1/G2 机制字段完整，质量通过，端到端指标改善 | `accept` | 扩展请求分布并做回归测试 |
| 有吞吐或接受率收益，但 verify 成本、TTFT 或尾延迟仍需改善 | `tune` | 调整 proposal length、draft 或候选预算 |
| unsupported、OOM、质量失败或性能退化 | `reject` | 保留失败 JSON，回到 G0 或更换候选 |


## 相关阅读

完成本节 benchmark 后，可以继续比较缓存复用和请求调度；跨项目比较时，沿用相同的 workload、延迟、吞吐和质量记录方式。下面的论文与开源文档用于对照接受规则、真实 backend 和工程实现。
- [69. Prefix Caching Benchmark | 前缀缓存基准](./69_Prefix_Caching_Benchmark.ipynb)
- [70. Serving Scheduler Benchmark | 推理服务调度基准](./70_Serving_Scheduler_Benchmark.ipynb)
- [Fast Inference from Transformers via Speculative Decoding（论文）](https://arxiv.org/abs/2211.17192)
- [Accelerating Large Language Model Decoding with Speculative Sampling（论文）](https://arxiv.org/abs/2302.01318)
- [vLLM Speculative Decoding（开源实现文档）](https://docs.vllm.ai/en/latest/examples/features/speculative_decoding/)
- [SGLang Speculative Decoding（开源实现文档）](https://docs.sglang.ai/advanced_features/speculative_decoding.html)